In [5]:
!pip install -q transformers datasets langchain_core langchain_huggingface bitsandbytes accelerate>=0.26.0 hf_transfer

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFacePipeline
from huggingface_hub import login

import torch, os, json, re, csv
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm

In [ ]:
huggingface_token = ""

from huggingface_hub import login
login(token=huggingface_token)

In [3]:
model_name = "K-intelligence/Midm-2.0-Mini-Instruct"
validation_file_path = "./validation.jsonl"
code_mapping_path = "./interior_codes.csv"  # 업로드된 매핑 파일 경로(변경 가능)

system_prompt = (    
    "너는 인테리어 전문가다. 주어진 조건에 맞는 인테리어 방법을 설명하라. 모든 답변은 반드시 한국어로만 작성한다. "
    "다음 제약을 절대적으로 준수하라: "
    "1) 7문장 이상 작성할 것. "
    "2) 숫자, 평 수, 지역 명, 아파트 명을 포함하지 말 것. "
    "3) JSON, 리스트, 코드블록, 영어 문장을 출력하지 말 것. "
    "4) 오직 자연스러운 한국어 문단만 작성할 것."
)

In [4]:
CODE_PATTERN = re.compile(r"\b[a-z]{3}_[0-9]{4}\b", re.IGNORECASE)

def load_code_mapping(csv_path: str):
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    cols_lower = [c.lower() for c in df.columns]
    pick = None
    for a,b in [("code","label"), ("code","name"), ("코드","라벨"), ("코드","이름")]:
        if a in cols_lower and b in cols_lower:
            pick = (df.columns[cols_lower.index(a)], df.columns[cols_lower.index(b)])
            break
    if pick is None:
        pick = (df.columns[0], df.columns[1])
    code_col, label_col = pick
    df[code_col] = df[code_col].astype(str).str.strip()
    df[label_col] = df[label_col].astype(str).str.strip()
    return dict(zip(df[code_col], df[label_col]))

def replace_codes(text: str, mapping: dict):
    if not text:
        return text
    def repl(m):
        code = m.group(0)
        return mapping.get(code, code)
    return CODE_PATTERN.sub(repl, text)

In [5]:
def model_load(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=huggingface_token, use_fast=False)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    hf_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        token=huggingface_token,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

    pipe = pipeline("text-generation", model=hf_model, tokenizer=tokenizer, max_new_tokens=1024)
    model = HuggingFacePipeline(pipeline=pipe)
    return model, tokenizer

def create_chain(model_name):
    model, tokenizer = model_load(model_name)
    chat = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "{instruction}"}
    ]
    prompt_str = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    prompt = PromptTemplate.from_template(prompt_str)
    chain = prompt | model | StrOutputParser()
    return chain

In [6]:
def load_jsonl_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line=line.strip()
            if line:
                data.append(json.loads(line))
    return data

def extract_after_assistant(full_text: str):
    if not full_text:
        return ""
    tag = "<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
    pos = full_text.find(tag)
    if pos == -1:
        return full_text.strip()
    start = pos + len(tag)
    if start < len(full_text) and full_text[start:start+1] == "\n":
        start += 1
    return full_text[start:].strip()

In [7]:
chain = create_chain(model_name)
dataset = load_jsonl_data(validation_file_path)
code_map = load_code_mapping(code_mapping_path)

print(f"로드된 데이터 개수: {len(dataset)}")
print("첫 번째 데이터 예시:")
print("Input:", dataset[0]["input"])
print("Output:", dataset[0]["output"])

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


로드된 데이터 개수: 135
첫 번째 데이터 예시:
Input: 제목: 레이아웃변경으로 실용적인 인테리어 완성 / 태그: 우물천장,간접우물천장,대공원월드메르디앙아파트인테리어,거실인테리어,실링팬인테리어,한샘다이닝룸,다이닝룸인테리어,한샘안방인테리어,드레스룸포켓도어,한샘욕실인테리어안방욕실인테리어대형욕실인조대리석탑볼탑볼세면대,한샘자녀방인테리어아치인테리어아치디자인자녀방,한샘한샘욕실거실욕실욕실인테리어욕실디자인,한샘현관인테리어,중문인테리어 / 공간: spa_0008,spa_0001,spa_0009,spa_0006,spa_0010,spa_0002,spa_0007,spa_0011 / 스타일: sty_0004 / 예산: cos_0009
Output: 주방 ㅣ 11자 주방의 심플한 디자인 (feat.비스포크 키친핏냉장고) 자 녀 방 ㅣ 두개의 공간을 통합과 분리를 통해 효율적 공간 디자인.


In [9]:
# sample_num = 3
# c = 0

# for i in range(len(dataset)):
#     question_raw = dataset[i]["input"]
#     # 1) 입력 전 치환: 코드 → 라벨
#     question_pre = replace_codes(question_raw, code_map)

#     full_result = chain.invoke({"instruction": question_pre})
#     assistant_response = extract_after_assistant(full_result)

#     # 2) 출력 후 보정(안전망): 혹시 모델이 코드를 출력했으면 다시 라벨로
#     assistant_response = replace_codes(assistant_response, code_map)

#     c += 1
#     print("--------------------------------------------------------------")
#     print("### Question(raw): ", question_raw)
#     print("### Question(pre): ", question_pre)
#     # print("\n### Full Answer: ", full_result)
#     print("\n### Extracted Assistant Response: ", assistant_response)
#     print("\n### Expected Output: ", dataset[i]["output"])
#     if c == sample_num:
#         break

--------------------------------------------------------------
### Question(raw):  제목: 레이아웃변경으로 실용적인 인테리어 완성 / 태그: 우물천장,간접우물천장,대공원월드메르디앙아파트인테리어,거실인테리어,실링팬인테리어,한샘다이닝룸,다이닝룸인테리어,한샘안방인테리어,드레스룸포켓도어,한샘욕실인테리어안방욕실인테리어대형욕실인조대리석탑볼탑볼세면대,한샘자녀방인테리어아치인테리어아치디자인자녀방,한샘한샘욕실거실욕실욕실인테리어욕실디자인,한샘현관인테리어,중문인테리어 / 공간: spa_0008,spa_0001,spa_0009,spa_0006,spa_0010,spa_0002,spa_0007,spa_0011 / 스타일: sty_0004 / 예산: cos_0009
### Question(pre):  제목: 레이아웃변경으로 실용적인 인테리어 완성 / 태그: 우물천장,간접우물천장,대공원월드메르디앙아파트인테리어,거실인테리어,실링팬인테리어,한샘다이닝룸,다이닝룸인테리어,한샘안방인테리어,드레스룸포켓도어,한샘욕실인테리어안방욕실인테리어대형욕실인조대리석탑볼탑볼세면대,한샘자녀방인테리어아치인테리어아치디자인자녀방,한샘한샘욕실거실욕실욕실인테리어욕실디자인,한샘현관인테리어,중문인테리어 / 공간: 전체,거실,침실,아이방,키친,다이닝룸,욕실,현관 / 스타일: 심플&미니멀 / 예산: 7천만원 이상

### Extracted Assistant Response:  인테리어를 실용적으로 구성하기 위해 레이아웃 변경과 다양한 디자인 요소를 활용하는 방법을 설명드리겠습니다.

먼저, 공간의 용도에 따라 레이아웃을 재구성하는 것이 중요합니다. 예를 들어, 거실의 경우에는 가족 구성원들이 편안하게 머무를 수 있도록 공간의 흐름을 자연스럽게 구성하고, 다양한 수납 공간을 추가하여 실용성을 높일 수 있습니다. 또한, 우물천장이나 간접우물천장을 활용하여 공간을 더 넓어 보이게 하고, 조명을 통해 분위기를 연출할 수 있습니다.

침실 인테리어에서는 공간의 기능성을 고려하여 

In [10]:
code_map = load_code_mapping(code_mapping_path)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = f"eval_results_min_preprocessed_{ts}.csv"

with open(save_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["extracted_response", "expected_output"])
    writer.writeheader()

    for i in tqdm(range(len(dataset)), desc="Generating with Preprocessed Inputs"):
        # 입력(raw) 확보: title 우선, 없으면 text/그 외 dict → 문자열 변환
        raw_input = dataset[i].get("input", "")
        if isinstance(raw_input, dict):
            question_raw = raw_input.get("title") or raw_input.get("text") or json.dumps(raw_input, ensure_ascii=False)
        else:
            question_raw = str(raw_input)

        # (사전 처리) 코드→라벨 치환
        question_pre = replace_codes(question_raw, code_map)

        # LLM 호출
        try:
            full_result = chain.invoke({"instruction": question_pre})
        except Exception as e:
            full_result = f"[ERROR] {type(e).__name__}: {e}"

        # Qwen 템플릿에서 본문만 추출
        extracted = extract_after_assistant(full_result)

        # 정답(기대 출력)
        expected = dataset[i].get("output", "")

        # 최소 컬럼 저장
        writer.writerow({
            "extracted_response": extracted,
            "expected_output": expected
        })

print(f"✅ CSV 저장 완료: {save_path}")

Generating with Preprocessed Inputs:   0%|          | 0/135 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✅ CSV 저장 완료: eval_results_min_preprocessed_20251021_184239.csv


In [ ]:
midm